# 1. Train predictions

Trains the 45 models (LSTM, XGBoost and k-NN, each on its own, with the GMM regime gate, and pooled across all 40 stocks, at daily, weekly and monthly horizons) and saves their up-probabilities to `Data/Predictions_45/`. This is the slow notebook; a GPU helps the LSTM.

In [ ]:
# imports, output folder, and the model grid
# all imports, the output folder, and the model grid
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

DATA_DIR = "Data"

NEW = f"{DATA_DIR}/Predictions_45"     # only prediction parquets are saved here - no model files
os.makedirs(NEW, exist_ok=True)

CUTOFF = "2023-01-01"
GMM_K = 3                        # regimes only; no hi/lo/min_regime_acc - thresholds are chosen later
MODEKEY = {("daily", "single"): "", ("weekly", "together"): "wk_", ("weekly", "split"): "split_wk_",
           ("monthly", "together"): "mo_", ("monthly", "split"): "split_mo_"}
MODES = [("daily", "single"), ("weekly", "together"), ("weekly", "split"),
         ("monthly", "together"), ("monthly", "split")]

ALGOS = ["lstm", "xgb", "knn"]
SCALE_FUSIONS = [("daily", ["single"]), ("weekly", ["together", "split"]), ("monthly", ["together", "split"])]
TIERS = ["bare", "gmm", "pooled"]
STARTS = ["2000-01-01", "2010-01-01", "2015-01-01"]
XDAYS = [10, 20, 30, 100]
print("new predictions ->", NEW)

In [ ]:
# read the features table (two parquet parts)
FEATURES = ["logret1", "vol20", "logret20", "range20", "dd60", "hl_range", "close_chg", "open_chg"]
INPUT_COLS = FEATURES + ["Close", "Open"]

def read_features(parts):
    long = pd.concat([pd.read_parquet(p) for p in parts]).set_index("Date")
    return {t: g.drop(columns="ticker") for t, g in long.groupby("ticker")}

feats = read_features([f"{DATA_DIR}/features_1.parquet", f"{DATA_DIR}/features_2.parquet"])
print(len(feats), "tickers")

In [ ]:
# look at data (+ plot)
# Look at the data - change TICKER, see its table and plot the price + one feature.
TICKER = "AAPL"

df = feats[TICKER].sort_index()
print(f"{TICKER}: {len(df)} rows, {df.index.min().date()} -> {df.index.max().date()}, up-rate {df['y_up'].mean():.3f}")
display(df[["Open", "High", "Low", "Close"] + FEATURES + ["y_up"]].tail())
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(df.index, df["Close"]); ax[0].set_ylabel("Close"); ax[0].set_title(TICKER)
ax[1].plot(df.index, df["vol20"], color="tab:orange"); ax[1].set_ylabel("vol20")
plt.tight_layout(); plt.show()

In [ ]:
# data health
# Data health - row counts per ticker, NaNs in the model inputs, and the up/down balance.
sizes = pd.Series({t: len(d) for t, d in feats.items()})
print("tickers:", len(feats), "| rows/ticker:", int(sizes.min()), "..", int(sizes.max()))
print("shortest 5:", dict(sizes.nsmallest(5)))
print("total NaNs in INPUT_COLS (all tickers):", int(sum(feats[t][INPUT_COLS].isna().sum().sum() for t in feats)), "(should be 0)")
up = pd.Series({t: feats[t]["y_up"].mean() for t in feats})
print(f"daily up-rate: mean {up.mean():.3f}, range {up.min():.3f}..{up.max():.3f}")
display(pd.concat([feats[t][FEATURES] for t in list(feats)[:5]]).describe().loc[["mean", "std", "min", "max"]].T.round(3))

In [ ]:
# Windows - anchors + builder (+ shared prep)
# Windows + the two prep helpers the runners AND the inspects share, so what you inspect is exactly
# what the grid runs: fit_scaler (train-only StandardScaler) and windows_for (scale -> windows + masks).
def week_starts(dates):
    iso = dates.isocalendar()
    wid = iso["year"].to_numpy() * 100 + iso["week"].to_numpy()
    return np.where(np.r_[True, wid[1:] != wid[:-1]])[0]

def month_starts(dates):
    ym = dates.year.to_numpy() * 100 + dates.month.to_numpy()
    return np.where(np.r_[True, ym[1:] != ym[:-1]])[0]

SCALES = {"daily": None, "weekly": week_starts, "monthly": month_starts}

def default_y(scale, fusion):
    if scale == "daily": return 0
    return 6 if (scale == "monthly" and fusion == "together") else 5

def make_windows(Z, close, dates, x_days, scale, fusion, y):
    day, anc, one, yy, at, nxt = [], [], [], [], [], []
    if scale == "daily":
        for t in range(x_days - 1, len(dates) - 1):
            one.append(Z[t - x_days + 1:t + 1])
            yy.append(int(close[t + 1] > close[t])); at.append(t); nxt.append(t + 1)
    else:
        A = SCALES[scale](dates)
        for j in range(len(A) - 1):
            t, tn = A[j], A[j + 1]
            s = t - x_days + 1
            if s < 0:
                continue
            if fusion == "split":
                if j < y - 1:
                    continue
                day.append(Z[s:t + 1]); anc.append(Z[A[j - y + 1:j + 1]])
            else:
                k = np.searchsorted(A, s)
                if k < y:
                    continue
                one.append(np.vstack([Z[A[k - y:k]], Z[s:t + 1]]))
            yy.append(int(close[tn] > close[t])); at.append(t); nxt.append(tn)
    yy, at, nxt = np.array(yy, "float32"), np.array(at), np.array(nxt)
    X = [np.array(day, "float32"), np.array(anc, "float32")] if fusion == "split" else [np.array(one, "float32")]
    return X, yy, at, nxt

def fit_scaler(df, train_start, cutoff):
    dates = df.index
    m = (dates >= pd.Timestamp(train_start)) & (dates < pd.Timestamp(cutoff))
    return StandardScaler().fit(df.loc[m, INPUT_COLS])

def windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y):
    dates, close = df.index, df["Close"].to_numpy()
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    en = pd.Timestamp(end) if end is not None else dates.max()
    Z = scaler.transform(df[INPUT_COLS])
    X, y_all, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
    tr = (dates[at] >= ts) & (dates[nxt] < co)
    te = (dates[at] >= co) & (dates[at] <= en)
    return Z, X, y_all, at, nxt, tr, te

def frame(df, dates, at, y, prob):
    out = df.loc[dates[at], ["Open", "High", "Low", "Close"]].copy()
    out["y_up"], out["prob_up"] = y, prob
    return out

def flat(X):
    return X.reshape(len(X), -1)

_POOL = {}

In [ ]:
# windows (clear example)
# Clear windows example - runs the real windows_for on one ticker with a tiny x_days, then reads off
# the last few actual windows and their next-period label.
TICKER, SCALE, FUSION, X_DAYS = "AAPL", "daily", "single", 3

df = feats[TICKER].sort_index()
scaler = fit_scaler(df, "2015-01-01", CUTOFF)
Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, "2015-01-01", CUTOFF, None, SCALE, FUSION, X_DAYS, default_y(SCALE, FUSION))
dates, close = df.index, df["Close"].to_numpy()
print(f"{TICKER} {SCALE}: {len(y_all)} windows; last 5 (label = next period up?):")
for i in list(range(len(y_all)))[-5:]:
    a = at[i]
    seq = ", ".join(f"{dates[k].date()}:{close[k]:.2f}" for k in range(a - X_DAYS + 1, a + 1))
    print(f"  window {i}: [{seq}] -> {dates[nxt[i]].date()} close {close[nxt[i]]:.2f}  y_up={int(y_all[i])}")
print("real model input X[i] shape:", X[0].shape[1:], "= (steps, n_features)")

In [ ]:
# weekly / monthly cutting
# Weekly / monthly cutting - pick a ticker + scale and see how the daily series is cut into periods:
# the period start (first trading day of each week/month), how many days fall in it, and how each period
# becomes one model input + a next-period label. Set SCALE to "weekly" or "monthly".
TICKER, SCALE, FUSION, X_DAYS = "AAPL", "monthly", "together", 10
Y = default_y(SCALE, FUSION)

df = feats[TICKER].sort_index()
dates, close = df.index, df["Close"].to_numpy()
A = SCALES[SCALE](dates)                                   # index of the first trading day of each period
print(f"{TICKER}: {len(dates)} daily bars -> {len(A)} {SCALE} periods (predict period-start to next period-start).")
display(pd.DataFrame({"period_start": dates[A].date,
                      "days_in_period": np.r_[np.diff(A), len(dates) - A[-1]],
                      "start_Close": close[A].round(2)}).tail(8))

scaler = fit_scaler(df, "2015-01-01", CUTOFF)
Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, "2015-01-01", CUTOFF, None, SCALE, FUSION, X_DAYS, Y)
print(f"\nlast 5 windows ({FUSION} = "
      + (f"{Y} prior anchors + {X_DAYS} daily bars" if FUSION == "together" else f"{X_DAYS} daily bars & {Y} prior anchors")
      + "), label = next period up?")
for i in list(range(len(y_all)))[-5:]:
    print(f"  {dates[at[i]].date()} -> {dates[nxt[i]].date()}: close {close[at[i]]:.2f} -> {close[nxt[i]]:.2f}  y_up={int(y_all[i])}")
print("model input shape(s):", [x.shape[1:] for x in X])

In [ ]:
# no leakage (purge)
# No-leakage (purge) check - uses the real windows_for masks: last train target < cutoff, first test
# window at/after it, zero overlap.
TICKER, SCALE, FUSION, X_DAYS = "AAPL", "daily", "single", 10

df = feats[TICKER].sort_index()
scaler = fit_scaler(df, "2015-01-01", CUTOFF)
Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, "2015-01-01", CUTOFF, None, SCALE, FUSION, X_DAYS, default_y(SCALE, FUSION))
dates, co = df.index, pd.Timestamp(CUTOFF)
print(f"{TICKER} {SCALE}: train {int(tr.sum())}, test {int(te.sum())} windows")
print("last train target:", dates[nxt[tr]].max().date(), "-> before cutoff:", bool(dates[nxt[tr]].max() < co))
print("first test anchor:", dates[at[te]].min().date(), "-> at/after cutoff:", bool(dates[at[te]].min() >= co))
print("train/test overlap:", int((tr & te).sum()), "(must be 0)")

In [ ]:
# Regime - fit the GMM (regime / GMM rank)
# fit_gmm fits the GMM on the train features; regime_at re-derives the regime for a config's test rows
# (scaler + windows + GMM, no model training) so the +GMM tier can reuse the bare prob_up.
def fit_gmm(Z, dates, train_start, cutoff, k, seed):
    m = (dates >= pd.Timestamp(train_start)) & (dates < pd.Timestamp(cutoff))
    return GaussianMixture(k, covariance_type="full", n_init=5, random_state=seed).fit(Z[m])

def regime_at(ticker, feats, train_start, cutoff, end, scale, fusion, x_days, gmm_k, seed=42):
    df = feats[ticker].sort_index()
    scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, default_y(scale, fusion))
    gm = fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    return gm.predict(Z[at[te]])

In [ ]:
# GMM regime profile
# What each GMM regime is - per-regime average of every feature (raw units) + regime share by year.
TICKER = "AAPL"

df = feats[TICKER].sort_index()
scaler = fit_scaler(df, "2015-01-01", CUTOFF)
Z = scaler.transform(df[INPUT_COLS])
reg = fit_gmm(Z, df.index, "2015-01-01", CUTOFF, GMM_K, 42).predict(Z)
print(f"{TICKER}: per-regime mean of each feature (raw units):")
display(df[INPUT_COLS].assign(regime=reg).groupby("regime").mean().round(4))
print("regime share by year (rows sum to 1):")
display(pd.crosstab(df.index.year, reg, normalize="index").round(2).tail(10))

In [ ]:
# data prep (one example)
# Inspect one example - runs the real prep (fit_scaler + windows_for + fit_gmm) for one ticker, so you
# see each step exactly as the grid computes it. Change TICKER / SCALE / FUSION / X_DAYS.
TICKER, SCALE, FUSION, X_DAYS = "AAPL", "monthly", "together", 10
Y = default_y(SCALE, FUSION)

df = feats[TICKER].sort_index()
print(f"=== {TICKER} · {SCALE} · {FUSION} · x_days={X_DAYS} · y={Y} ===")
print("\n1) RAW features (before scaling), tail:")
display(df[INPUT_COLS].tail())

scaler = fit_scaler(df, "2015-01-01", CUTOFF)
Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, "2015-01-01", CUTOFF, None, SCALE, FUSION, X_DAYS, Y)
print("2) SCALED (train-fit StandardScaler, mean~0/std~1), same rows:")
display(pd.DataFrame(Z, index=df.index, columns=INPUT_COLS).tail())

if SCALE != "daily":
    A = SCALES[SCALE](df.index)
    print(f"3) {SCALE.upper()} anchors - first trading day of each period ({len(A)} total); last few:")
    display(df.iloc[A[-6:]][["Open", "Close"]])

print("4) Windows:", [a.shape for a in X], "| labels:", y_all.shape,
      "| train/test:", int(tr.sum()), "/", int(te.sum()), "| up-rate:", round(float(y_all.mean()), 3))
print("   last training window (real model input, scaled):")
display(pd.DataFrame(X[0][tr][-1], columns=INPUT_COLS))

gm = fit_gmm(Z, df.index, "2015-01-01", CUTOFF, GMM_K, 42)
print("5) GMM regime at anchors - counts:", dict(pd.Series(gm.predict(Z[at])).value_counts().sort_index()))

In [ ]:
# LSTM - network + fit
# LSTM network + fit (in memory). Nothing is saved to disk - only the predictions go to parquet.
def build_lstm(steps, n_feat, u1=256, u2=128, dropout=0.01, lr=1e-3):
    m = Sequential([Input((steps, n_feat)),
                    LSTM(u1, return_sequences=True), Dropout(dropout),
                    LSTM(u2), Dropout(dropout),
                    Dense(1, activation="sigmoid")])
    m.compile(optimizer=Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

def steps_of(x_days, scale, fusion, y, branch):
    if scale == "daily":  return x_days
    if fusion == "split": return x_days if branch == 0 else y
    return x_days + y

def _cb():
    return [EarlyStopping("val_accuracy", mode="max", patience=25, restore_best_weights=True),
            ReduceLROnPlateau("val_loss", factor=0.5, patience=5, min_lr=1e-6)]

def fit_lstm(X, y, steps, x_days, scale, epochs=200, batch=16, val_frac=0.1):
    cut = int(len(X) * (1 - val_frac))
    purge = (x_days - 1) if scale == "daily" else x_days // 5
    m = build_lstm(steps, X.shape[2])
    m.fit(X[:cut - purge], y[:cut - purge], validation_data=(X[cut:], y[cut:]),
          epochs=epochs, batch_size=batch, verbose=0, callbacks=_cb())
    return m

def predict_lstm(models, Xlist):
    return np.mean([m.predict(a, verbose=0).ravel() for m, a in zip(models, Xlist)], axis=0)

In [ ]:
# LSTM - pooled (one model over 40)
# Pooled LSTM - one model over all 40 tickers; memoised per (scale, fusion, start, x_days).
def pooled_lstm(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, val_frac=0.1):
    key = ("lstm", scale, fusion, train_start, cutoff, x_days, gmm_k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    np.random.seed(seed); tf.random.set_seed(seed)
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    purge = (x_days - 1) if scale == "daily" else x_days // 5
    Xf, Xv = [[] for _ in range(n)], [[] for _ in range(n)]
    yf, yv = [], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        cut = int(m.sum() * (1 - val_frac))
        if cut - purge < 1 or m.sum() - cut < 1:
            continue
        for i in range(n):
            Xi = X[i][m]; Xf[i].append(Xi[:cut - purge]); Xv[i].append(Xi[cut:])
        yy_m = yy[m]; yf.append(yy_m[:cut - purge]); yv.append(yy_m[cut:])
    yf, yv = np.concatenate(yf), np.concatenate(yv)
    models = []
    for i in range(n):
        m = build_lstm(steps_of(x_days, scale, fusion, y, i), np.vstack(Xf[i]).shape[2])
        m.fit(np.vstack(Xf[i]), yf, validation_data=(np.vstack(Xv[i]), yv),
              epochs=200, batch_size=16, verbose=0, callbacks=_cb())
        models.append(m)
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# LSTM - run_lstm
# run_lstm - bare / +GMM / pooled, any of the five modes. Shares fit_scaler + windows_for with inspect.
def run_lstm(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
             x_days=10, y=None, gmm=False, pooled=False, seed=42, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    np.random.seed(seed); tf.random.set_seed(seed)
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_lstm(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [fit_lstm(X[i][tr], y_all[tr], steps_of(x_days, scale, fusion, y, i), x_days, scale)
                  for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_lstm(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_lstm(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# LSTM on every mode (no save)
# Try LSTM on every mode for one ticker (uses run_lstm). Trains in memory and writes nothing to disk;
# just numbers to compare. (Trains a net per mode, so it is slower than XGBoost / k-NN.)
TICKER = "AAPL"
for scale, fusion in MODES:
    r, _ = run_lstm(TICKER, feats, "2015-01-01", CUTOFF, scale=scale, fusion=fusion, gmm=True, gmm_k=GMM_K)
    print(f"{scale:8s} {fusion:9s}: {len(r):4d} rows | AUC {roc_auc_score(r['y_up'].astype(int), r['prob_up']):.4f}"
          f" | regimes {dict(r['regime'].value_counts().sort_index())}")

In [ ]:
# XGBoost - core + pooled
# XGBoost core - params, tabular prediction, pooled trainer (memoised).
XGB_P = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=5,
             eval_metric="logloss", tree_method="hist", n_jobs=-1)

def predict_tab(models, Xlist):
    return np.mean([m.predict_proba(flat(a))[:, 1] for m, a in zip(models, Xlist)], axis=0)

def pooled_xgb(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k):
    key = ("xgb", scale, fusion, train_start, cutoff, x_days, gmm_k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    Xf, yf = [[] for _ in range(n)], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        if m.sum() == 0:
            continue
        for i in range(n): Xf[i].append(flat(X[i][m]))
        yf.append(yy[m])
    Xf = [np.vstack(a) for a in Xf]; yf = np.concatenate(yf)
    models = [XGBClassifier(random_state=seed, **XGB_P).fit(Xf[i], yf.astype(int)) for i in range(n)]
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# XGBoost - run_xgb
# run_xgb.
def run_xgb(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
            x_days=10, y=None, gmm=False, pooled=False, seed=42, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    np.random.seed(seed)
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_xgb(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [XGBClassifier(random_state=seed, **XGB_P).fit(flat(X[i][tr]), y_all[tr].astype(int)) for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_tab(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_tab(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# XGBoost on every mode (no save)
# Try XGBoost on every mode for one ticker (uses run_xgb). Trains in memory, saves nothing - just numbers.
TICKER = "AAPL"
for scale, fusion in MODES:
    r, _ = run_xgb(TICKER, feats, "2015-01-01", CUTOFF, scale=scale, fusion=fusion, gmm=True, gmm_k=GMM_K)
    print(f"{scale:8s} {fusion:9s}: {len(r):4d} rows | AUC {roc_auc_score(r['y_up'].astype(int), r['prob_up']):.4f}"
          f" | regimes {dict(r['regime'].value_counts().sort_index())}")

In [ ]:
# k-NN - core + pooled
# k-NN core (k=50) - pooled index (memoised).
def pooled_knn(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, k=50):
    key = ("knn", scale, fusion, train_start, cutoff, x_days, gmm_k, k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    Xf, yf = [[] for _ in range(n)], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        if m.sum() == 0:
            continue
        for i in range(n): Xf[i].append(flat(X[i][m]))
        yf.append(yy[m])
    Xf = [np.vstack(a) for a in Xf]; yf = np.concatenate(yf)
    models = [KNeighborsClassifier(k).fit(Xf[i], yf.astype(int)) for i in range(n)]
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# k-NN - run_knn
# run_knn.
def run_knn(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
            x_days=10, y=None, gmm=False, pooled=False, seed=42, k=50, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_knn(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [KNeighborsClassifier(k).fit(flat(X[i][tr]), y_all[tr].astype(int)) for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_tab(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_tab(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# k-NN on every mode (no save)
# Try k-NN on every mode for one ticker (uses run_knn). Trains in memory, saves nothing - just numbers.
TICKER = "AAPL"
for scale, fusion in MODES:
    r, _ = run_knn(TICKER, feats, "2015-01-01", CUTOFF, scale=scale, fusion=fusion, gmm=True, gmm_k=GMM_K)
    print(f"{scale:8s} {fusion:9s}: {len(r):4d} rows | AUC {roc_auc_score(r['y_up'].astype(int), r['prob_up']):.4f}"
          f" | regimes {dict(r['regime'].value_counts().sort_index())}")

In [ ]:
# Driver - train_tier (check parquet, append)
# One tier at a time: read its combined parquet, train only the (ticker, start, x_days) configs that
# are NOT already in it, and append them back to the SAME parquet. The gmm tier reuses the bare prob_up
# (same LSTM) and only adds the regime - no retraining. Writes after each block, so a crash resumes.
RUNNERS = {"lstm": run_lstm, "xgb": run_xgb, "knn": run_knn}

def train_tier(algo, scale, fusion, tier):
    modek = MODEKEY[(scale, fusion)]
    path = f"{NEW}/{tier}_{modek}{algo}_test_preds.parquet"
    have = pd.read_parquet(path) if os.path.exists(path) else pd.DataFrame()
    done = set(zip(have["ticker"], have["train_start"].astype(str).str[:10], have["x_days"].astype(int))) if len(have) else set()
    bpath = f"{NEW}/bare_{modek}{algo}_test_preds.parquet"
    bare = pd.read_parquet(bpath) if (tier == "gmm" and os.path.exists(bpath)) else None
    parts, n_new = ([have] if len(have) else []), 0
    for xd in XDAYS:
        for st in STARTS:
            block = []
            for tk in sorted(feats):
                if (tk, st, int(xd)) in done:
                    continue
                if tier == "gmm" and bare is not None:
                    m = (bare["ticker"] == tk) & (bare["train_start"].astype(str).str[:10] == st) & (bare["x_days"].astype(int) == int(xd))
                    if m.any():
                        out = bare.loc[m, ["ticker", "y_up", "prob_up", "train_start", "x_days", "scale", "fusion"]].reset_index(drop=True)
                        out.insert(3, "regime", regime_at(tk, feats, st, CUTOFF, None, scale, fusion, xd, GMM_K))
                        out["tier"] = "gmm"
                        block.append(out); n_new += 1
                        continue
                kw = dict(scale=scale, fusion=fusion, x_days=xd, gmm_k=GMM_K)
                if tier == "gmm":      kw["gmm"] = True
                elif tier == "pooled": kw["pooled"] = True
                res, _ = RUNNERS[algo](tk, feats, st, CUTOFF, **kw)
                keep = ["y_up", "prob_up"] + (["regime"] if tier != "bare" else [])
                out = res[keep].reset_index(drop=True)
                out.insert(0, "ticker", tk)
                out["train_start"], out["x_days"] = st, int(xd)
                out["scale"], out["fusion"], out["tier"] = scale, fusion, tier
                block.append(out); n_new += 1
            if block:
                parts += block
                pd.concat(parts, ignore_index=True).to_parquet(path)
    return n_new

In [ ]:
# Run - run_matrix()
# Loop the 45 tiers. Each reads its own parquet and trains only the configs missing from it - LSTM,
# XGBoost and k-NN all the same way. Whatever is already in Predictions_45 is kept (adds 0). Re-runnable.
def run_matrix():
    tiers = [(a, s, f, t) for a in ALGOS for s, fus in SCALE_FUSIONS for f in fus for t in TIERS]
    for i, (a, s, f, t) in enumerate(tiers, 1):
        n = train_tier(a, s, f, t)
        print(f"[{i:2d}/{len(tiers)}] {a:4s} {s}/{f} {t:6s}: +{n} trained")
    print("done")

# run_matrix()

In [ ]:
# training progress
# Progress - configs present in each combined parquet vs the full 40x3x4 = 480 per tier.
rows = []
for algo in ALGOS:
    for scale, fus in SCALE_FUSIONS:
        for fusion in fus:
            for tier in TIERS:
                path = f"{NEW}/{tier}_{MODEKEY[(scale, fusion)]}{algo}_test_preds.parquet"
                if os.path.exists(path):
                    d = pd.read_parquet(path, columns=["ticker", "train_start", "x_days"])
                    n = d.drop_duplicates().shape[0]
                else:
                    n = 0
                rows.append({"algo": algo, "mode": f"{scale}/{fusion}", "tier": tier, "done": n, "of": 480})
prog = pd.DataFrame(rows)
print("tiers complete:", int((prog.done >= 480).sum()), "/ 45")
display(prog)

In [ ]:
# Universal - one parquet, all 45
# One universal parquet - stack all 45 tier files into a single file (with an `algo` column) so the
# data-analysis step reads one thing. bare tiers have no `regime` (NaN there); everything else lines up.
frames = []
for algo in ALGOS:
    for scale, fus in SCALE_FUSIONS:
        for fusion in fus:
            for tier in TIERS:
                path = f"{NEW}/{tier}_{MODEKEY[(scale, fusion)]}{algo}_test_preds.parquet"
                if os.path.exists(path):
                    frames.append(pd.read_parquet(path).assign(algo=algo))
if frames:
    allp = pd.concat(frames, ignore_index=True)
    allp.to_parquet(f"{NEW}/all_predictions.parquet")
    print("universal:", allp.shape, "->", f"{NEW}/all_predictions.parquet")
    display(allp.groupby(["algo", "scale", "fusion", "tier"]).size().rename("rows").reset_index())
else:
    print("no tier parquets found yet - run the training cells first")

In [ ]:
# read the universal parquet
# Read the universal parquet and eyeball it - shape, memory, columns, and rows per tier.
U = pd.read_parquet(f"{NEW}/all_predictions.parquet")
print("shape:", U.shape, "| memory:", round(U.memory_usage(deep=True).sum() / 1e6, 1), "MB")
print("columns:", list(U.columns))
display(U.head())
display(U.groupby(["algo", "scale", "fusion", "tier"]).size().rename("rows").reset_index())

In [ ]:
# filter the universal (raw)
# Filter - slice the universal parquet with plain pandas (change the values and re-run).
U = pd.read_parquet(f"{NEW}/all_predictions.parquet")
one = U[(U.algo == "lstm") & (U.scale == "monthly") & (U.fusion == "together") & (U.tier == "pooled")
        & (U.ticker == "AAPL") & (U.train_start == "2015-01-01") & (U.x_days == 10)]
print("one slice (algo/mode/tier/ticker/start/x_days):", len(one), "rows")
display(one.head())
print("rows by tier:", U.groupby("tier").size().to_dict())
print("prob_up >= 0.60:", int((U.prob_up >= 0.60).sum()),
      "| <= 0.40:", int((U.prob_up <= 0.40).sum()),
      "| between 0.45-0.55:", int(U.prob_up.between(0.45, 0.55).sum()))

In [ ]:
# sort - raw AUC per tier
# Sort - a raw first look before the analysis notebook: directional AUC + accuracy per tier, best first.
U = pd.read_parquet(f"{NEW}/all_predictions.parquet")
lb = []
for k, g in U.groupby(["algo", "scale", "fusion", "tier"]):
    y = g["y_up"].astype(int)
    lb.append({**dict(zip(["algo", "scale", "fusion", "tier"], k)), "n": len(g),
               "auc": round(roc_auc_score(y, g["prob_up"]), 4) if y.nunique() > 1 else np.nan,
               "acc@0.5": round(((g["prob_up"] >= 0.5).astype(int) == y).mean(), 4)})
lb = pd.DataFrame(lb).sort_values("auc", ascending=False)
display(lb.head(15))